In [46]:
from IPython.display import clear_output
import warnings
warnings.filterwarnings('ignore')
clear_output(wait=True)

In [47]:
clear_output(wait=True)
print('hello world')

hello world


In [48]:
import time 
for i in range(5):
    clear_output(wait=True)
    print(f"Step {i+1} completed")
    time.sleep(1)

Step 5 completed


In [49]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

from IPython.display import display, clear_output
import pandas as pd 

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


print('librarires imported successfully!!')


librarires imported successfully!!


In [50]:
# stop spark session is there any 
try : 
    spark.stop()
    print('stopped previous spark session!!')

except : 
    pass 

# start new session 

try: 
    from pyspark import SparkContext
    sc = SparkContext._active_spark_context
    if sc : 
        sc.stop()
        print('stopped previous sparkContext !!')
except : 
    pass 


spark = SparkSession.builder\
    .appName('02-Single aggregation')\
    .master('spark://spark-master:7077')\
    .config("spark.executor.memory", "3g") \
    .config("spark.driver.memory", "4g") \
    .config('spark.sql.adaptive.enabled', 'true')\
    .config('spark.sql.debug.maxToStringFields', '1000')\
    .getOrCreate()

spark.conf.set('spark.sql.debug.maxToStringFields', '1000')

print('Spark started successfully !!')
print(f'Spark version : {spark.version}')

stopped previous spark session!!
Spark started successfully !!
Spark version : 3.5.1


### Step 2-a : Load Fire Departement Data 

In [52]:
# Define the schema 

fire_schema = StructType([StructField('CallNumber', IntegerType(), True),
StructField('UnitID', StringType(), True),
StructField('IncidentNumber', IntegerType(), True),
StructField('CallType', StringType(), True),                  
StructField('CallDate', StringType(), True),      
StructField('WatchDate', StringType(), True),
StructField('CallFinalDisposition', StringType(), True),
StructField('AvailableDtTm', StringType(), True),
StructField('Address', StringType(), True),       
StructField('City', StringType(), True),       
StructField('Zipcode', IntegerType(), True),       
StructField('Battalion', StringType(), True),                 
StructField('StationArea', StringType(), True),       
StructField('Box', StringType(), True),       
StructField('OriginalPriority', StringType(), True),       
StructField('Priority', StringType(), True),       
StructField('FinalPriority', IntegerType(), True),       
StructField('ALSUnit', BooleanType(), True),       
StructField('CallTypeGroup', StringType(), True),
StructField('NumAlarms', IntegerType(), True),
StructField('UnitType', StringType(), True),
StructField('UnitSequenceInCallDispatch', IntegerType(), True),
StructField('FirePreventionDistrict', StringType(), True),
StructField('SupervisorDistrict', StringType(), True),
StructField('Neighborhood', StringType(), True),
StructField('Location', StringType(), True),
StructField('RowID', StringType(), True),
StructField('Delay', FloatType(), True)])

# Set up the data source file's path 
sf_fire_file = '/opt/spark-data/sf-fire-calls.csv'

# load dataframe with San Francesco fire data 

fire_df = spark.read.csv(sf_fire_file, header= True, schema = fire_schema)
        #.option('samplingRatio', 0.00001)\ # to infere schema

print(' --------- data is loaded successfully --------- ')
# print(f'Number of rows is : {fire_df.count()}')
# print(f'Number of columns is {len(fire_df.columns)}')

 --------- data is loaded successfully --------- 


### Step 2 : Prepare Data For Parquet Storage 
Before aggrefatioins, let's prepare the data : 
1. clean column names (remove spaces from partition values)
2. write as partitioned parquet 
3. Compare CSV vs Parquet performance and metadata 

In [ ]:
# Create clean partition columns (no spaces, no special caracters)
# Also extract data for data-based partitionning 
# Replace problematic caracters for file system compatibility : 
#   - spaces -> underscores
#   - forward slash -> underscore (can cause directory issues)
#   - parentheses -> removed (can cause shell issues)

fire_df_clean = fire_df.withColumn(
        'CallType_Clean', 
        regexp_replace(
            regexp_replace(
                regexp_replace(
                    col('CallType'), r"[\s/]", "_"
                ), 
                r"[()]", ""
            ),
            r"_+", "_"
        )
    )\
    .withColumn(
        'Neighborhood_Clean',
        regexp_replace(
            regexp_replace(
                col('Neighborhood'),
                r"[\s/]", "_"
            ),
            r"_+", "_"            
        )
    )\
    .withColumn(
        'call_date',
        to_date(col('CallDate'), 'MM/dd/yyyy')
    )


print('created clean partition columns')
print('Original VS Clean column examples :')

result = fire_df_clean.select(
    'CallDate', 'call_date',
    'CallType', 'CallType_Clean',
    'Neighborhood', 'Neighborhood_Clean'

).distinct()

pdf = result.limit(20).toPandas()

display(pdf)


### Step 2-b : Write Partitioned Parquet Files 
Parquet advantages : 
- Columnar storage - faster for analytics
- Compression - smaller file size 
- Schema embedded - no need to define schema when reading 
- Partitionning - faster question on partition columns 

In [ ]:
print('Writing multilevel partioned parquet file ...')
print('Hierarchy : Date -> CallType -> Priority')
print()

fire_df_clean.write.mode('overwrite')\
    .partitionBy('call_date', 'CallType_Clean', 'Priority')\
    .parquet('/opt/spark-data/output/fire-aggregations-partitionned')

print('parquet filres written to : /opt/spark-data/output/fire-aggregations-partitionned/')

### Step 2c : Compare CSV vs Parquet Metadata 

In [ ]:
# compare file size 
import os 
import subprocess 

# get csv file size 
csv_size = os.path.getsize('/opt/spark-data/sf-fire-calls.csv')
print('--- CSV FILE ---')
print(f'Size : {csv_size} bytes, {csv_size / (1024**2):.2f} MB')

# Get Parquet Directory Size 
try : 
    result = subprocess.run(['du', '-sh', '/opt/spark-data/output/fire-aggregations-partitionned/'],
                            #capture_output = True, text = True )
    parquet_size_str = result.stdout.split()[0]
    print(' --- PARQUET FILES --- ')
    print(f"Size : {parquet_size_str}")
    
except : 
    print('\n Parquet Files : output/fire-aggregations-partitionned/')


#count parquet files 
try : 
    result2 = subprocess.run(['find', '/opt/spark-data/output/fire-aggregations-partitionned/', '-name', '*.parquet', '-type', 'f'],
                         capture_output = True, text = True)
    parquet_files = len(result2.stdout.strip().split('\n'))
    print(f'Number of partition files : {parquet_files}')
except : 
    pass 

### CSV Format : 
- Text based (larger size)
- Row oriented (slower for analytics)
- Must define schema when reading
- No compression
  
### Parquet Format : 
- Binary columnar (smaller size)
- Column oriented (faster for analytics)
- Schema auto-detected
- Build-compression
- Partition pruning (skip irrelevant data)

In [ ]:
#read parquet (schema is auto-detected)
parquet_df = spark.read.parquet('/opt/spark-data/output/fire-aggregations-partitionned')

print(f'Loaded from parquet : {parquet_df.count():,} rows')
print(f'Schema auto-detected from file (no manual definition needed)')
print(f'Total Columns : {len(parquet_df.columns)}')

In [ ]:
# Show parquet schema 
print('Parquet schema (auto-detected):')
parquet_df.printSchema()


print('\n Notice : ') 
print(' - Include partition columns : call_date, CallType_clean, Priority')
print(' - Preserve data types correctly ')
print(' - No manula schema definitoin needed')

### Step 2e: Demonstrate Partition Pruning 

Show how partitioned Parquet files skip irrelevant data 

In [ ]:
print('Query: Medical incidents in January 2021 with Priority 3')
print('Parquet reads only : call_date = 2021-01-* / CallType_clean = Medical_Incidents / Priority=3/')
print()

result3 = parquet_df.filter("year(call_date)=2018 and month(call_date)=1 and CallType_Clean = 'Medical_Incident' and Priority='3'")


print(f' Rows found : {result3.count():,}')
print('Partitioning Benifits :')
print('- Skill all other dates')
print('- Skip all other call types ')
print('- Skip priorities 1 and 2 ')
print('- Read only ~3% of total data ')


In [ ]:
# Example : Date Range Query 

jan_2018_data = parquet_df.filter(
    (col('call_date') >= '2018-01-01') &
    (col('call_date') < '2018-02-01') 
)
print(f' Rows found : {jan_2018_data.count():,}')


### Step 2f : 

Now we use the parquet data for all remaining aggregations


In [ ]:
# Use Parquet DataFrame for all aggregations
fire_df = parquet_df

print("✓ Using Parquet data for aggregations")
print(f"Total rows: {fire_df.count()}")
print(f"Total columns: {len(fire_df.columns)}")


### Step 3 : Single Aggregation 

In [8]:

# Count rows
print("Count total rows:")
total_calls = fire_df.count()
print(f" Total emergency calls: {total_calls:,}")



Count total rows:


[Stage 0:>                                                        (0 + 11) / 11]

 Total emergency calls: 175,296


In [10]:
# Average of a column (example: Delay)
print("Calculate average response delay:")
result = fire_df.select(avg("Delay").alias("Average_Delay_Minutes"))
pdf = result.toPandas()
display(pdf)




Calculate average response delay:


,Average_Delay_Minutes
0,3.892364


In [11]:
# Min, Max, Avg together
print("Find min and max delay:")
result = fire_df.select(
    min("Delay").alias("Fastest_Response"),      # Smallest delay
    max("Delay").alias("Slowest_Response"),      # Largest delay
    avg("Delay").alias("Average_Response")       # Average delay
)
pdf = result.toPandas()
display(pdf)

Find min and max delay:


,Fastest_Response,Slowest_Response,Average_Response
0,0.016667,1844.550049,3.892364


In [12]:
# agg() - Applies multiple aggregation functions at once
# count() - Counts rows
# avg() - Average value
# min() - Minimum value
# max() - Maximum value
# stddev() - Standard deviation (how spread out the values are)

print("Multiple statistics together:")
result = fire_df.agg(
    count("*").alias("Total_Calls"),        # Count all rows
    avg("Delay").alias("Avg_Delay"),        # Average delay
    min("Delay").alias("Min_Delay"),        # Smallest delay
    max("Delay").alias("Max_Delay"),        # Largest delay
    stddev("Delay").alias("Std_Dev_Delay")  # How varied the delays are
)

pdf = result.limit(20).toPandas()
display(pdf)


Multiple statistics together:


,Total_Calls,Avg_Delay,Min_Delay,Max_Delay,Std_Dev_Delay
0,175296,3.892364,0.016667,1844.550049,9.378286


### Step 4: Group By - Single Column 

In [13]:
# groupBy() - Groups rows that have the same value in a column
# count() - Counts rows in each group
# orderBy() - Sorts the results
# desc() - Descending order (highest first)

print("Count calls by CallType:")
call_type_counts = fire_df.groupBy("CallType") \
    .agg(count("*").alias("Number_of_Calls")) \
    .orderBy(desc("Number_of_Calls"))

pdf = call_type_counts.limit(20).toPandas()
display(pdf)


Count calls by CallType:


,CallType,Number_of_Calls
0,Medical Incident,113794
1,Structure Fire,23319
2,Alarms,19406
3,Traffic Collision,7013
4,Citizen Assist / Service Call,2524
5,Other,2166
6,Outside Fire,2094
7,Vehicle Fire,854
8,Gas Leak (Natural and LP Gases),764
9,Water Rescue,755


In [19]:
# Count calls by Neighborhood 

print('Count calls by Neighborhood')
neighborhood_counts = fire_df.groupBy('Neighborhood')\
    .agg(count('*').alias('Total_Calls'))\
    .orderBy(col('Total_Calls').desc())
pdf = neighborhood_counts = neighborhood_counts.limit(20).toPandas()
display(pdf)

Count calls by Neighborhood


,Neighborhood,Total_Calls
0,Tenderloin,22785
1,South of Market,16623
2,Mission,15912
3,Financial District/South Beach,11912
4,Bayview Hunters Point,9616
5,Sunset/Parkside,6984
6,Western Addition,6591
7,Nob Hill,5810
8,Outer Richmond,4604
9,Hayes Valley,4357


In [20]:
# Count calls by Priority level
print("Count calls by Priority level:")
priority_counts = fire_df.groupBy("Priority") \
    .agg(count("*").alias("Number_of_Calls")) \
    .orderBy("Priority")

pdf = priority_counts.limit(20).toPandas()
display(pdf)

Count calls by Priority level:


,Priority,Number_of_Calls
0,1,6473
1,2,29748
2,3,131166
3,A,144
4,B,66
5,C,8
6,E,7689
7,I,2


### Step 5: Multiple Aggregation on Grouped Data 

You can calculater many things at once for each group 

In [21]:
# groupBy() - Group by CallType
# agg() - Apply multiple aggregations:
# count() - How many calls of each type
# avg() - Average delay for each type
# min() - Fastest response for each type
# max() - Slowest response for each type
# stddev() - Variation in response times

print("Call type with count, avg delay, min, max:")
call_type_stats = fire_df.groupBy("CallType").agg(
    count("*").alias("call_count"),              # Total calls per type
    avg("Delay").alias("avg_delay"),             # Average delay per type
    min("Delay").alias("min_delay"),             # Fastest response per type
    max("Delay").alias("max_delay"),             # Slowest response per type
    stddev("Delay").alias("delay_stddev")        # Consistency of response times
).orderBy(desc("call_count"))

pdf = call_type_stats.limit(20).toPandas()
display(pdf)


Call type with count, avg delay, min, max:


,CallType,call_count,avg_delay,min_delay,max_delay,delay_stddev
0,Medical Incident,113794,3.874868,0.016667,721.400024,4.762796
1,Structure Fire,23319,3.679561,0.183333,1370.233276,18.149547
2,Alarms,19406,3.542729,0.150000,1844.550049,13.473035
3,Traffic Collision,7013,3.789132,0.016667,239.949997,5.264109
4,Citizen Assist / Service Call,2524,5.473343,0.100000,142.800003,6.232400
5,Other,2166,5.505155,0.016667,405.700012,11.984743
6,Outside Fire,2094,4.181948,0.116667,340.116669,10.200601
7,Vehicle Fire,854,3.903923,0.766667,62.833332,3.911556
8,Gas Leak (Natural and LP Gases),764,4.583399,1.183333,76.566666,4.697733
9,Water Rescue,755,5.507748,0.483333,82.849998,4.447013


In [22]:
# Neighborhood statistics 

print("Neighborhood statistics :")
neighborhood_stats = fire_df.groupBy("Neighborhood").agg(
    count("*").alias("call_count"),              # Total calls per type
    avg("Delay").alias("avg_delay"),             # Average delay per type
    min("Delay").alias("min_delay"),             # Fastest response per type
    max("Delay").alias("max_delay")             # Slowest response per type
).orderBy(desc("call_count"))

pdf = neighborhood_stats.limit(20).toPandas()
display(pdf)


Neighborhood statistics :


,Neighborhood,call_count,avg_delay,min_delay,max_delay
0,Tenderloin,22785,3.728089,0.066667,340.483337
1,South of Market,16623,4.004297,0.066667,1844.550049
2,Mission,15912,3.813656,0.166667,949.116638
3,Financial District/South Beach,11912,3.888582,0.150000,628.616638
4,Bayview Hunters Point,9616,4.150425,0.116667,734.049988
5,Sunset/Parkside,6984,3.901821,0.233333,199.449997
6,Western Addition,6591,3.691820,0.016667,87.216667
7,Nob Hill,5810,3.695404,0.083333,184.516663
8,Outer Richmond,4604,3.939053,0.233333,106.333336
9,Hayes Valley,4357,3.771161,0.233333,332.950012


### Step 6: Group by Multiple columns 

Group by two or more columns.

In [23]:
# Neighborhood and CallType statistics 

print("Neighborhood statistics :")
neighborhood_calltype = fire_df.groupBy("Neighborhood", "CallType").agg(
    count("*").alias("call_count"),              # Total calls per type
).orderBy(desc("call_count"))

pdf = neighborhood_calltype.limit(20).toPandas()
display(pdf)


Neighborhood statistics :


,Neighborhood,CallType,call_count
0,Tenderloin,Medical Incident,17109
1,South of Market,Medical Incident,12181
2,Mission,Medical Incident,10796
3,Financial District/South Beach,Medical Incident,6532
4,Bayview Hunters Point,Medical Incident,6099
5,Sunset/Parkside,Medical Incident,4936
6,Western Addition,Medical Incident,4191
7,Nob Hill,Medical Incident,3516
8,Outer Richmond,Medical Incident,3075
9,Hayes Valley,Medical Incident,2731


In [26]:
# Count by Priority AND CallType
print("Count by Priority AND CallType:")
priority_calltype = fire_df.groupBy("Priority", "CallType") \
    .agg(count("*").alias("call_count")) \
    .orderBy("Priority", desc("call_count"))

pdf = priority_calltype.limit(20).toPandas()
display(pdf)


Count by Priority AND CallType:


,Priority,CallType,call_count
0,1,Medical Incident,6374
1,1,Traffic Collision,96
2,1,Administrative,2
3,1,Other,1
4,2,Medical Incident,27401
5,2,Traffic Collision,2116
6,2,Other,134
7,2,Citizen Assist / Service Call,86
8,2,Alarms,4
9,2,Structure Fire,2


### Step 7: Conditional Aggregation 

Count rows that match sprcifc conditions. Like counting : "how many blue books?"

In [25]:
print("count fast vs slow responses bu CallType:")
response_speed = fire_df.groupBy('CallType').agg(
    count('*').alias('Total_calls'),
    sum(when(col('Delay')<5, 1).otherwise(0)).alias('fast_responses'),
    sum(when(col('Delay')<10, 1).otherwise(0)).alias('normal_responses'),
    sum(when(col('Delay')>=0, 1).otherwise(0)).alias('slow_responses')
).orderBy(desc('total_calls'))

pdf = response_speed.limit(10).toPandas()
display(pdf)

count fast vs slow responses bu CallType:


,CallType,Total_calls,fast_responses,normal_responses,slow_responses
0,Medical Incident,113794,94636,110267,113794
1,Structure Fire,23319,20878,22609,23319
2,Alarms,19406,17581,19266,19406
3,Traffic Collision,7013,5806,6726,7013
4,Citizen Assist / Service Call,2524,1713,2298,2524
5,Other,2166,1573,1981,2166
6,Outside Fire,2094,1786,2016,2094
7,Vehicle Fire,854,728,829,854
8,Gas Leak (Natural and LP Gases),764,595,740,764
9,Water Rescue,755,421,693,755


In [27]:
# Count high priority vs low priority by Neighborhood
print("Count high priority vs low priority by Neighborhood:")
priority_by_neighborhood = fire_df.groupBy("Neighborhood").agg(
    count("*").alias("total_calls"),
    sum(when(col("Priority") == "3", 1).otherwise(0)).alias("high_priority"),
    sum(when(col("Priority") == "2", 1).otherwise(0)).alias("medium_priority"),
    sum(when(col("Priority") == "1", 1).otherwise(0)).alias("low_priority")
).orderBy(desc("total_calls"))

pdf = priority_by_neighborhood.limit(20).toPandas()
display(pdf)


Count high priority vs low priority by Neighborhood:


,Neighborhood,total_calls,high_priority,medium_priority,low_priority
0,Tenderloin,22785,16430,4202,1109
1,South of Market,16623,11961,3174,767
2,Mission,15912,11495,2996,639
3,Financial District/South Beach,11912,9266,1899,389
4,Bayview Hunters Point,9616,7250,1402,346
5,Sunset/Parkside,6984,5134,1190,281
6,Western Addition,6591,4974,1103,237
7,Nob Hill,5810,4441,948,187
8,Outer Richmond,4604,3436,774,174
9,Hayes Valley,4357,3293,736,151


### Step 8: Count Distinct Values 
countDistinct : count diffrent values(ignore duplicates). Like counting : "How many different authors?" not "How many books?"

In [28]:
# countDistinct = count unique values (ignore duplicates).
# Different from count() which counts all rows including duplicates.

# Count unique call types
print("Count unique call types:")
result = fire_df.agg(countDistinct("CallType").alias("Unique_Call_Types"))

pdf = result.limit(20).toPandas()
display(pdf)


Count unique call types:


,Unique_Call_Types
0,30


In [29]:
# Count unique values for multiple columns
print("Count unique values for multiple columns:")
result = fire_df.agg(
    countDistinct("CallType").alias("Unique_CallTypes"),        # Different call types
    countDistinct("Neighborhood").alias("Unique_Neighborhoods"),# Different neighborhoods
    countDistinct("Battalion").alias("Unique_Battalions"),      # Different battalions
    countDistinct("Priority").alias("Unique_Priorities")        # Different priorities
)

pdf = result.limit(20).toPandas()
display(pdf)

Count unique values for multiple columns:


,Unique_CallTypes,Unique_Neighborhoods,Unique_Battalions,Unique_Priorities
0,30,42,11,8


In [32]:
# List all distinct call types
print("List all distinct call types:")
result = fire_df.select("CallType").distinct().orderBy("CallType")

pdf = result.limit(20).toPandas()
display(pdf)


List all distinct call types:


,CallType
0,Administrative
1,Aircraft Emergency
2,Alarms
3,Assist Police
4,Citizen Assist / Service Call
5,Confined Space / Structure Collapse
6,Electrical Hazard
7,Elevator / Escalator Rescue
8,Explosion
9,"Extrication / Entrapped (Machinery, Vehicle)"


### Step 9: Filter Aggregated Results 

In [35]:
# Show only neighborhoods with more than 5,000 calls
print("Show only neighborhoods with more than 5,000 calls:")
busy_neighborhoods = fire_df.groupBy("Neighborhood") \
    .agg(count("*").alias("total_calls")) \
    .filter(col("total_calls") > 5000) \
    .orderBy(desc("total_calls"))

pdf = busy_neighborhoods.limit(20).toPandas()
display(pdf)


Show only neighborhoods with more than 5,000 calls:


,Neighborhood,total_calls
0,Tenderloin,22785
1,South of Market,16623
2,Mission,15912
3,Financial District/South Beach,11912
4,Bayview Hunters Point,9616
5,Sunset/Parkside,6984
6,Western Addition,6591
7,Nob Hill,5810


In [36]:
# Show call types with average delay > 10 minutes
print("Show call types with average delay > 10 minutes:")
slow_call_types = fire_df.groupBy("CallType") \
    .agg(
        count("*").alias("call_count"),
        avg("Delay").alias("avg_delay")
    ) \
    .filter(col("avg_delay") > 10) \
    .orderBy(desc("avg_delay"))

pdf = slow_call_types.limit(20).toPandas()
display(pdf)


Show call types with average delay > 10 minutes:


,CallType,call_count,avg_delay
0,Mutual Aid / Assist Outside Agency,9,38.416666
1,Assist Police,35,26.981904
2,Train / Rail Incident,57,16.452047
3,Administrative,3,12.261111


### Step 10: Sort Aggregated Data 

In [37]:
# asc() - Ascending order (smallest to largest: 1, 2, 3...)
# Useful for finding items with the smallest values first
print("Sort by avg_delay (ascending = fastest first):")
result = fire_df.groupBy("CallType") \
    .agg(
        count("*").alias("call_count"),
        avg("Delay").alias("avg_delay")
    ) \
    .orderBy(asc("avg_delay"))  # Sort by average delay, fastest first

pdf = result.limit(20).toPandas()
display(pdf)



Sort by avg_delay (ascending = fastest first):


,CallType,call_count,avg_delay
0,Alarms,19406,3.542729
1,Structure Fire,23319,3.679561
2,Aircraft Emergency,36,3.773148
3,Traffic Collision,7013,3.789132
4,Medical Incident,113794,3.874868
5,Vehicle Fire,854,3.903923
6,Explosion,89,4.110674
7,Outside Fire,2094,4.181948
8,Elevator / Escalator Rescue,453,4.337822
9,"Extrication / Entrapped (Machinery, Vehicle)",28,4.391667


In [38]:
# Sort by multiple columns
print("Sort by multiple columns:")
result = fire_df.groupBy("Neighborhood", "Priority") \
    .agg(count("*").alias("call_count")) \
    .orderBy("Neighborhood", desc("call_count"))

pdf = result.limit(20).toPandas()
display(pdf)


Sort by multiple columns:


,Neighborhood,Priority,call_count
0,Bayview Hunters Point,3,7250
1,Bayview Hunters Point,2,1402
2,Bayview Hunters Point,E,597
3,Bayview Hunters Point,1,346
4,Bayview Hunters Point,A,19
5,Bayview Hunters Point,B,2
6,Bernal Heights,3,2424
7,Bernal Heights,2,529
8,Bernal Heights,E,197
9,Bernal Heights,1,105


### Step 11: Calculate Percentages 

Show what percentage each group represents.

In [39]:
# Percentage of calls by CallType
print("Percentage of calls by CallType:")

# First, get counts
call_type_counts = fire_df.groupBy("CallType") \
    .agg(count("*").alias("call_count"))

# Get total
total = fire_df.count()

# Calculate percentage
call_type_pct = call_type_counts.withColumn(
    "percentage",
    (col("call_count") / total * 100)
).orderBy(desc("percentage"))

pdf = call_type_pct.limit(20).toPandas()
display(pdf)


Percentage of calls by CallType:


,CallType,call_count,percentage
0,Medical Incident,113794,64.915343
1,Structure Fire,23319,13.302642
2,Alarms,19406,11.070418
3,Traffic Collision,7013,4.000662
4,Citizen Assist / Service Call,2524,1.439850
5,Other,2166,1.235624
6,Outside Fire,2094,1.194551
7,Vehicle Fire,854,0.487176
8,Gas Leak (Natural and LP Gases),764,0.435834
9,Water Rescue,755,0.430700


### Step 12: Real-World Analysis Example 
Questions:Which neighborhood needs more resources?
- Hight call volume
- Slow average rspose times
- Many hign-priority calls

In [42]:
# Percentage of calls by CallType
print("Neighborhood resource analysis:")
 
resource_analysis = fire_df.groupBy("Neighborhood").agg(
    count("*").alias("total_calls"),
    avg('Delay').alias("avg_response_time"),
    sum(when(col('Priority')=='3', 1).otherwise(0)).alias('high_priority_calls'),
    sum(when(col('Delay')>15 , 1).otherwise(0)).alias('very_slow_responses')
).orderBy(desc("total_calls"))

print("Neighborhoods that need more resources:")

pdf = resource_analysis.limit(20).toPandas()
display(pdf)


Neighborhood resource analysis:
Neighborhoods that need more resources:


,Neighborhood,total_calls,avg_response_time,high_priority_calls,very_slow_responses
0,Tenderloin,22785,3.728089,16430,320
1,South of Market,16623,4.004297,11961,265
2,Mission,15912,3.813656,11495,235
3,Financial District/South Beach,11912,3.888582,9266,185
4,Bayview Hunters Point,9616,4.150425,7250,178
5,Sunset/Parkside,6984,3.901821,5134,107
6,Western Addition,6591,3.691820,4974,96
7,Nob Hill,5810,3.695404,4441,72
8,Outer Richmond,4604,3.939053,3436,75
9,Hayes Valley,4357,3.771161,3293,58


In [44]:
problem_areas = resource_analysis.filter(
    (col("total_calls") > 3000) &
    (col("avg_response_time") > 4) 
).orderBy(desc('avg_response_time'))


print('Problem areas (high volumes + slow responses ):') 
pdf = problem_areas.limit(20).toPandas()
display(pdf)

Problem areas (high volumes + slow responses ):


,Neighborhood,total_calls,avg_response_time,high_priority_calls,very_slow_responses
0,Bayview Hunters Point,9616,4.150425,7250,178
1,Chinatown,3731,4.033199,2903,40
2,South of Market,16623,4.004297,11961,265
3,Bernal Heights,3260,4.004085,2424,57


26/10/07 15:13:52 ERROR TaskSchedulerImpl: Lost executor 8 on 172.19.0.5: worker lost: Not receiving heartbeat for 60 seconds
26/10/07 15:13:52 ERROR TaskSchedulerImpl: Lost executor 11 on 172.19.0.3: worker lost: Not receiving heartbeat for 60 seconds
26/10/07 15:13:52 ERROR TaskSchedulerImpl: Lost executor 9 on 172.19.0.6: worker lost: Not receiving heartbeat for 60 seconds


In [43]:
resource_analysis.select(max('avg_response_time')).show()

+----------------------+
|max(avg_response_time)|
+----------------------+
|      5.47149999499321|
+----------------------+



### Window Functions 

In [53]:
spark.stop()
print("The spark session have ended")

The spark session have ended
